[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C25_Long_Context_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与长上下文热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy **从零实现**每条长上下文技术，再与朴素参考 **对拍**。

这个 notebook 做三件事：① 确认环境；② 用真实数字感受**长上下文的四道墙**（位置 / 内存 / 算力 / KV）到底有多陡；③ 立下全课的纪律——**对拍（differential testing）**。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画热力图/曲线）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 内存墙：注意力的 n×n 分数矩阵有多大？

朴素注意力要 materialize 一个 `n×n` 的分数矩阵 `S`。算一笔账：**单个注意力头**的 S（FP16，2 字节）在不同序列长度下多大？对比一张 80GB 的 H100，看朴素注意力在什么长度就**单头都放不下**了。

In [ ]:
def score_matrix_gb(n, bytes_per=2):
    '''单个头的 n×n 分数矩阵大小（GB）。FP16=2 字节，1GB=1e9 字节。'''
    return bytes_per * n * n / 1e9

H100_GB = 80
print(f"{'序列长 n':>10s} {'单头 S (GB)':>14s} {'占 H100 比例':>14s}")
for n in [4096, 32768, 131072, 1_000_000]:
    gb = score_matrix_gb(n)
    print(f'{n:>10d} {gb:>14.3f} {gb/H100_GB:>13.1%}')
# 4k 时才 33MB，128k 时单头就要 34GB，1M 时单头 2000GB —— 这就是内存墙
assert score_matrix_gb(131072) > 30, '128k 单头分数矩阵应 >30GB'
assert score_matrix_gb(1_000_000) > H100_GB, '1M 单头远超 H100'
print('\n✅ 内存墙：n×n 随 n 平方膨胀，128k 单头就放不下 → 模块 02 FlashAttention 不存它')

## 3 · 算力墙：注意力计算量随 n 平方

注意力的 FLOPs 主要来自 `QKᵀ` 和 `P·V`，都是 `O(n²·d)`。算一下从 4k 到更长，计算量放大了多少倍——这解释了为什么模块 03/04 要把复杂度从 O(n²) 降到 O(n·W) 甚至 O(n)。

In [ ]:
def attn_flops(n, d=128):
    # QK^T: n*n*d 次乘加 ; P@V: n*n*d ; 合计 ~ 2*n^2*d 的 2 倍(乘+加)
    return 2 * 2 * n * n * d

base = attn_flops(4096)
print(f"{'序列长 n':>10s} {'相对 4k 的计算量':>18s}")
for n in [4096, 16384, 65536, 262144]:
    print(f'{n:>10d} {attn_flops(n)/base:>17.0f}x')
# n 翻 4 倍 -> 计算量翻 16 倍（平方）
assert abs(attn_flops(8192)/attn_flops(4096) - 4.0) < 1e-9, 'n 翻倍 -> 计算量 4 倍'
print('\n✅ 算力墙：n 平方增长。64k 比 4k 慢 256 倍 → 模块 03/04 降复杂度')

## 4 · KV 墙：KV cache 随长度线性膨胀

自回归解码要缓存每个历史 token 的 K、V。缓存大小 = `2(K和V) × 层数 × KV头数 × head_dim × n × 字节`。用 Llama-2-7B 量级的配置算一下，KV cache 在长上下文下吃掉多少显存。

In [ ]:
def kv_cache_gb(n, n_layers=32, n_kv_heads=32, head_dim=128, bytes_per=2):
    # 2 = K 和 V 各一份
    return 2 * n_layers * n_kv_heads * head_dim * n * bytes_per / 1e9

print(f"{'序列长 n':>10s} {'KV cache (GB)':>16s}")
for n in [4096, 32768, 131072]:
    print(f'{n:>10d} {kv_cache_gb(n):>16.2f}')
# 线性于 n。32k 就要 ~17GB，128k ~68GB —— 解码时的 KV 墙
assert abs(kv_cache_gb(8192)/kv_cache_gb(4096) - 2.0) < 1e-9, 'KV 线性于 n'
g_mqa = kv_cache_gb(131072, n_kv_heads=1)   # MQA 只 1 个 KV 头
print(f'\n用 MQA(1 个 KV 头) 把 128k 的 KV 从 {kv_cache_gb(131072):.0f}GB 压到 {g_mqa:.2f}GB')
assert g_mqa < kv_cache_gb(131072), 'MQA 应显著减小 KV'
print('✅ KV 墙：随 n 线性增长 → 模块 05 KV 压缩（量化/淘汰），GQA/MQA 见 C20')

## 5 · 位置墙：训练没见过的位置（直觉热身）

RoPE 按位置把坐标旋转一个角度。训练时只见过位置 0..L_train，推理到更远位置时，旋转角会落到**训练时从未出现的区间**。先用最小例子看「位置→旋转角」，建立模块 01 的直觉：外推 = 把模型推到它没见过的角度。

In [ ]:
def rope_angle(pos, dim_pair=0, d=64, base=10000.0):
    '''第 dim_pair 对维度在位置 pos 处的旋转角（弧度）。'''
    theta = base ** (-2.0 * dim_pair / d)
    return pos * theta

L_train = 4096
# 最低频维度（转得最慢）在训练末尾才转了多少圈？
slow = rope_angle(L_train, dim_pair=31, d=64) / (2*np.pi)   # 圈数
fast = rope_angle(L_train, dim_pair=0,  d=64) / (2*np.pi)
print(f'训练长度 {L_train}：最高频维度转了 {fast:.0f} 圈，最低频维度只转了 {slow:.3f} 圈')
# 外推到 4x：最低频维度进入训练从没到过的角度区间
slow_4x = rope_angle(4*L_train, dim_pair=31, d=64) / (2*np.pi)
print(f'外推到 {4*L_train}：最低频维度转到 {slow_4x:.3f} 圈（训练只到 {slow:.3f} 圈）')
assert slow < 1.0 and slow_4x > slow, '低频维度外推进入未见区间'
print('\n✅ 位置墙：低频维度在训练时没转满，外推把它推到未见角度 → 模块 01 各种 scaling 来救')

## 6 · 立纪律：对拍（differential testing）

本课每个实现都要和一个**朴素参考**比对，标准是 `np.allclose(mine, ref, atol=...)`。先把这个工作流跑通：写一个朴素 softmax 当参考，再写一个「数值稳定」版当被测实现，对拍它们**数学等价**——这正是模块 02 online softmax 的伏笔。

In [ ]:
def softmax_naive(x):
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

def softmax_stable(x):
    m = x.max(axis=-1, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=-1, keepdims=True)

rng = np.random.default_rng(0)
x = rng.standard_normal((4, 8))
assert np.allclose(softmax_naive(x), softmax_stable(x), atol=1e-12)
print('对拍通过：稳定版与朴素版数值一致 ✅')

# 朴素版在大 logit 下溢出（长上下文里 logit 可能很大）
big = np.array([[800.0, 801.0, 802.0]])
assert np.isnan(softmax_naive(big)).any(), '朴素版应溢出为 nan'
assert not np.isnan(softmax_stable(big)).any()
print('朴素版溢出为 nan，稳定版正确 → 数学等价 ≠ 数值等价（模块 02/04 的关键）')

def check_allclose(name, got, ref, atol=1e-10):
    '''全课统一的对拍裁判。'''
    ok = np.allclose(got, ref, atol=atol)
    err = float(np.max(np.abs(np.asarray(got)-np.asarray(ref)))) if np.size(got) else 0.0
    print(f'[{name:<28}] allclose={ok}  max|err|={err:.2e}')
    assert ok, f'{name} 与参考不一致！'
    return ok

check_allclose('softmax_stable vs naive', softmax_stable(x), softmax_naive(x))
print('\n这就是全课的工作流：写实现 → 对拍朴素参考 → assert 兜底。')

✅ 检查全部通过即环境就绪、四道墙的量级心里有数、方法论到位。

**本课的契约**：你用 numpy 写出的每条长上下文技术，都会用 `np.allclose` 对拍朴素参考；结构正确则数值一致，数值一致则逻辑可迁移到 transformers/vLLM。

**接下来五个模块**：01 RoPE Scaling/YaRN（位置墙）→ 02 FlashAttention（内存墙 + IO）→ 03 稀疏/滑窗（算力墙）→ 04 线性/SSM（算力墙）→ 05 KV 压缩 + 评测（KV 墙 + 验真）。

下一站：**模块 01 · RoPE Scaling 与 YaRN**。